In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

from pathlib import Path
from ipyfilechooser import FileChooser
from ipywidgets import widgets
from IPython.display import display
from faim_ipa.utils import get_git_root
from matplotlib.cm import colors

sys.path.append(str(get_git_root()))

from source.s01_convert_to_zarr.create_selection_csv_utils import (
    get_experiment_widget,
)

## Choose the git-repo on the file-server

In [ ]:
fc = FileChooser(
    path=get_git_root(),
    layout=widgets.Layout(width="100%"),
    show_only_dirs=True,
    title="Select Root Directory",
)

In [ ]:
display(fc)

In [ ]:
root_dir = Path(fc.selected)

## Select the experiment

In [ ]:
exp = get_experiment_widget(root_dir)

In [ ]:
display(exp)

## Load measurements

In [ ]:
measurements_dir = root_dir / "processed_data" / exp.value.name / "s03_measurements"
m_files = list(measurements_dir.glob("*.csv"))

selection_csv = exp.value / "selection.csv"
selection_df = pd.read_csv(selection_csv)

measurements = pd.concat([pd.read_csv(f) for f in m_files])

selection_df["position"] = selection_df["position"].str.removesuffix(".zarr")

measurements = measurements.merge(
    selection_df[["position", "condition"]], left_on="id", right_on="position"
)

In [ ]:
measurements

### Exclude positions
Select the positions you want to exclude from the plotting.

In [ ]:
checkboxes = [
    widgets.Checkbox(value=False, description=value)
    for value in (measurements["id"] + ": " + measurements["condition"]).unique()
]
grid = widgets.GridBox(
    checkboxes, layout=widgets.Layout(grid_template_columns="repeat(5, 200px)")
)

In [ ]:
display(grid)

In [ ]:
max_time_point = 361

In [ ]:
excluded = [e.description.split(":")[0] for e in grid.children if e.value]
included_measurements = measurements[~measurements["id"].isin(excluded)]

included_measurements = included_measurements[
    included_measurements["time"] <= max_time_point
]

fig = plt.figure(figsize=(10, 6), dpi=300)
ax = sns.lineplot(included_measurements, x="time", y="mean_intensity", hue="condition")
ax.set_xlabel("Frame")
ax.set_ylabel("Mean Intensity")
ax.legend_.set_title("Condition");

### Save

In [ ]:
fig.savefig(
    measurements_dir.parent / "quantification.png", dpi=300, bbox_inches="tight"
)
included_measurements.to_csv(
    measurements_dir.parent / "quantification.csv", index_label=False
)

## Plot with Molt Annotations
This plot is only possible if you have done the molt annotation.

In [ ]:
measurements_dir = root_dir / "processed_data" / exp.value.name / "s03_measurements"
m_files = list(measurements_dir.glob("*.csv"))

selection_csv = exp.value / "selection.csv"
selection_df = pd.read_csv(selection_csv)

molt_measurements = pd.concat([pd.read_csv(f) for f in m_files])

selection_df["position"] = selection_df["position"].str.removesuffix(".zarr")

molt_measurements = molt_measurements.merge(
    selection_df[["position", "condition"]], left_on="id", right_on="position"
)
molt_csv = exp.value / "molt.csv"
molt_df = pd.read_csv(molt_csv)
molt_df["position"] = molt_df["position"].str.removesuffix(".zarr")
molt_measurements = molt_measurements.merge(molt_df, left_on="id", right_on="position")

In [ ]:
excluded = [e.description.split(":")[0] for e in grid.children if e.value]
molt_included_measurements = molt_measurements[~molt_measurements["id"].isin(excluded)]

molt_included_measurements = molt_included_measurements[
    molt_included_measurements["time"] <= max_time_point
]

In [ ]:
avg_molt_times = molt_included_measurements.groupby("condition")[
    ["m1", "m2", "m3", "m4"]
].mean()
fig = plt.figure(figsize=(10, 6), dpi=300)
ax = sns.lineplot(
    molt_included_measurements, x="time", y="mean_intensity", hue="condition"
)
for con, col in zip(avg_molt_times.T.columns, colors.TABLEAU_COLORS.keys()):
    ax.vlines(
        avg_molt_times.T[con],
        ymin=molt_included_measurements["mean_intensity"].quantile(0.01),
        ymax=molt_included_measurements["mean_intensity"].quantile(0.99),
        colors=col,
        linestyles="--",
    )
ax.set_xlabel("Frame")
ax.set_ylabel("Mean Intensity")
ax.legend_.set_title("Condition");

### Save

In [ ]:
fig.savefig(
    measurements_dir.parent / "molt-annotated-quantification.png",
    dpi=300,
    bbox_inches="tight",
)
molt_included_measurements.to_csv(
    measurements_dir.parent / "molt-annotated-quantification.csv", index_label=False
)

## Plot Molt Aligned
This plot is only possible if you have done the molt annotation.

In [ ]:
measurements_dir = root_dir / "processed_data" / exp.value.name / "s03_measurements"
m_files = list(measurements_dir.glob("*.csv"))

selection_csv = exp.value / "selection.csv"
selection_df = pd.read_csv(selection_csv)

molt_measurements = pd.concat([pd.read_csv(f) for f in m_files])

selection_df["position"] = selection_df["position"].str.removesuffix(".zarr")

molt_measurements = molt_measurements.merge(
    selection_df[["position", "condition"]], left_on="id", right_on="position"
)
molt_csv = exp.value / "molt.csv"
molt_df = pd.read_csv(molt_csv)
molt_df["position"] = molt_df["position"].str.removesuffix(".zarr")
molt_measurements = molt_measurements.merge(molt_df, left_on="id", right_on="position")

In [ ]:
def add_pseudo_time(frame, M1, M2, M3, M4):
    if frame < M1:
        return np.round(frame / M1, 2)
    elif (frame >= M1) and (frame < M2):
        return 1 + np.round((frame - M1) / (M2 - M1), 2)
    elif (frame >= M2) and (frame < M3):
        return 2 + np.round((frame - M2) / (M3 - M2), 2)
    elif (frame >= M3) and (frame < M4):
        return 3 + np.round((frame - M3) / (M4 - M3), 2)
    else:
        return 4 + np.round((frame - M4) / 10, 2)

In [ ]:
molt_measurements["pseudo_time"] = molt_measurements.apply(
    lambda r: add_pseudo_time(r["time"], r["m1"], r["m2"], r["m3"], r["m4"]), axis=1
)

In [ ]:
molt_measurements

In [ ]:
# m1 = 1, m2 = 2, m3 = 3, m4 = 4
max_pseudo_time_point = 4

In [ ]:
excluded = [e.description.split(":")[0] for e in grid.children if e.value]
molt_included_measurements = molt_measurements[~molt_measurements["id"].isin(excluded)]

molt_included_measurements = molt_included_measurements[
    molt_included_measurements["pseudo_time"] <= max_pseudo_time_point
]

fig = plt.figure(figsize=(10, 6), dpi=300)
ax = sns.lineplot(
    molt_included_measurements, x="pseudo_time", y="mean_intensity", hue="condition"
)
ax.set_xlabel("Pseudo-Time")
ax.set_ylabel("Mean Intensity")
ax.legend_.set_title("Condition");

### Save

In [ ]:
fig.savefig(
    measurements_dir.parent / "molt-quantification.png", dpi=300, bbox_inches="tight"
)
molt_included_measurements.to_csv(
    measurements_dir.parent / "molt-quantification.csv", index_label=False
)